# LABORATORY SESSION 1 – Neural Network Analysis of Fitbit Data

### 1. Imports
- **`import os`** → handling files
- **`import pandas as pd`** → data manipulation and analysis using DataFrames
- **`from datetime import datetime`** → to handle and convert date/time information 
- **`from tabulate import tabulate`** → allows formatting data as clean, readable tables for text output or logs


In [ ]:
import os
import pandas as pd
from datetime import datetime
from tabulate import tabulate

### 2. Helper Functions

To make our code modular and readable, we’ll define several helper functions that handle small, specific tasks such as formatting dates, numbers, and summarizing columns.

---

##### `format_date(date_str)`

This function tries to interpret strings as dates.  
Some Fitbit files store timestamps in different formats (with or without hours), so we check both.

---

##### `format_number(value)`

This function ensures numeric outputs (like averages or standard deviations)  
are printed with **three decimal places**.

---

##### `process_col(column, serie)`

This function calculates **descriptive statistics** for one column:
- Count of non-null values  
- Number of missing values  
- Number of distinct values  
- (If numeric) → min, max, mean, and standard deviation  

It returns a dictionary containing these statistics, ready to be turned into a table.

---

##### `log_row(text, fp)`

A simple utility that writes text to both:
- the **console** (so we see progress live),
- and a **log file** (to keep results saved).


In [ ]:
def format_date(date_str):
    ret = pd.NaT
    try:
        ret = datetime.strptime(date_str, '%m/%d/%Y %I:%M:%S %p')
    except ValueError:
        pass

    try:
        ret = datetime.strptime(date_str, '%m/%d/%Y')
    except ValueError:
        pass

    return ret

def format_number(value):
    if isinstance(value, (int, float)):
        return f"{value:.3f}"
    return value

def process_col(column, serie):
    row = {'Column': column}

    row.update({
        'Count': format_number(serie.count()),
        'Null Count': format_number(serie.isnull().sum()),
        'Distinct Count': format_number(serie.nunique())
    })

    if pd.api.types.is_numeric_dtype(serie):
      row.update({
          'Min': format_number(serie.min()),
          'Max': format_number(serie.max()),
          'Average': format_number(serie.mean()),
          'Std Dev': format_number(serie.std())
      })
    
    return row

def log_row(text, fp):
  print(text)
  fp.write(text + "\n")

## 3. Clean the data

Now that we have helper functions, we can define the main data-cleaning function.

### What it does:
1. Opens a report file (`table_statistics.txt`).
2. Loops through every `.csv` file in the dataset folder.
3. Loads the file into a DataFrame.
4. Detects and converts columns that look like dates.
5. Computes per-column statistics using `process_col()`.
6. Logs those statistics as a formatted table using `tabulate`.
7. Saves a cleaned version of the CSV into a new folder.


In [ ]:
def main(datadir, cleandir):
  with open('table_statistics.txt', 'w') as f:
    for filename in os.listdir(datadir):
      if not filename.endswith('.csv'): continue

      file_path = os.path.join(datadir, filename)
      df = pd.read_csv(file_path)
      
      log_row(f"Statistics for {filename}:", f)

      stats = []
      date_columns_modified = False
        
      for column in df.columns:
        if df[column].dtype == 'object' and df[column].apply(lambda x: isinstance(x, str)).all():
          try:
            date_column = df[column].apply(format_date)
            if not date_column.isnull().all():
              df[column] = date_column
          except ValueError:
            pass
   
        row = process_col(column, df[column])
        stats.append(row)
          
      log_row(tabulate(stats, headers="keys", tablefmt="grid"), f)

      new_file_path = os.path.join(cleandir, f"{filename}")
      df.to_csv(new_file_path, index=False)

      f.flush()

if __name__ == "__main__":
    data ="/Users/marianaseabra/Downloads/script-20251107/archive/full"
    clean = '/Users/marianaseabra/Downloads/script-20251107/archive/clean'

    if not os.path.exists(clean):
      os.makedirs(clean)

    main(data, clean)